# Chapitre 1 — Télécharger les données (clé Kaggle)

Ce notebook **extrait le jeu de données RSNA** via l'API Kaggle. C'est le
prérequis aux chapitres 2.5, 3, 4 et 5.

**Avant de lancer :**
1. Avoir accepté les règles de la compétition sur
   <https://www.kaggle.com/competitions/rsna-breast-cancer-detection/rules>
   (sinon Kaggle renvoie une erreur 403).
2. Les identifiants Kaggle doivent être disponibles : `docker-run.sh` monte
   `~/.kaggle` (avec `kaggle.json`) en lecture seule dans le conteneur. Sinon,
   renseigner `KAGGLE_USERNAME` / `KAGGLE_KEY` (voir `.env.example`).

Le jeu complet (images DICOM) fait **~314 Go** → téléchargement optionnel,
désactivé par défaut. On commence par les labels (`train.csv`, ~2 Mo).

In [ ]:
import os
# Vérifie les identifiants AVANT d'importer kaggle (l'import authentifie aussitôt).
os.environ.setdefault('KAGGLE_CONFIG_DIR', os.path.expanduser('~/.kaggle'))
have_json = os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json'))
have_env = bool(os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'))
assert have_json or have_env, (
    'Identifiants Kaggle absents : monte ~/.kaggle (kaggle.json) ou renseigne '
    'KAGGLE_USERNAME / KAGGLE_KEY (voir README).')
import kaggle  # déclenche l'authentification
print('Authentification Kaggle OK')

In [ ]:
import glob, zipfile

COMP = 'rsna-breast-cancer-detection'
DATA_DIR = os.path.expanduser('~/data/rsna')   # volume monté -> persistant
os.makedirs(DATA_DIR, exist_ok=True)

def _unzip_all(folder):
    """Décompresse puis supprime tous les .zip d'un dossier."""
    for z in glob.glob(os.path.join(folder, '*.zip')):
        print('décompression', os.path.basename(z))
        with zipfile.ZipFile(z) as zf:
            zf.extractall(folder)
        os.remove(z)

print('Cible :', DATA_DIR)

In [ ]:
# --- Labels seuls (léger) : de quoi explorer avant de tout télécharger ---
import pandas as pd
train_csv = os.path.join(DATA_DIR, 'train.csv')
if not os.path.exists(train_csv):
    kaggle.api.competition_download_file(COMP, 'train.csv', path=DATA_DIR, quiet=False)
    _unzip_all(DATA_DIR)
df = pd.read_csv(train_csv)
print('train.csv :', df.shape)
print('prévalence cancer :', round(df['cancer'].mean(), 4))
df.head()

## Mini-jeu réel — quelques patients en DICOM

Le dataset complet fait **~314 Go** : trop pour une démo. On télécharge donc seulement **~4 patients** (2 avec cancer, 2 sains), avec leurs 4 vues (L/R × CC/MLO), soit une quinzaine de DICOM (~50–80 Mo). Ces images servent de support concret au **chapitre 2.5** (prétraitement), qui les transforme réellement au format GMIC.

La sélection est déterministe (mêmes patients à chaque exécution) et le téléchargement reprend où il s'est arrêté (skip des fichiers déjà présents).

In [ ]:
# Sélection déterministe : patients ayant les 4 vues, 2 cancer + 2 sains.
SAMPLE_DIR = os.path.expanduser('~/data/rsna_sample')
os.makedirs(os.path.join(SAMPLE_DIR, 'train_images'), exist_ok=True)

NEED = {('L', 'CC'), ('L', 'MLO'), ('R', 'CC'), ('R', 'MLO')}
views_by_pid = df.groupby('patient_id').apply(
    lambda g: set(zip(g['laterality'], g['view'])))
complete = views_by_pid[views_by_pid.apply(NEED.issubset)].index
cancer_by_pid = df[df['patient_id'].isin(complete)].groupby('patient_id')['cancer'].max()
pos = list(cancer_by_pid[cancer_by_pid == 1].index[:2])
neg = list(cancer_by_pid[cancer_by_pid == 0].index[:2])
SAMPLE_PIDS = sorted(int(p) for p in pos + neg)
print('Patients échantillon :', SAMPLE_PIDS, '(2 cancer + 2 sains)')

sample_rows = df[df['patient_id'].isin(SAMPLE_PIDS)]
sample_rows.to_csv(os.path.join(SAMPLE_DIR, 'train.csv'), index=False)

for _, r in sample_rows.iterrows():
    pid, iid = int(r['patient_id']), int(r['image_id'])
    dst = os.path.join(SAMPLE_DIR, 'train_images', str(pid))
    os.makedirs(dst, exist_ok=True)
    if os.path.exists(os.path.join(dst, f'{iid}.dcm')):
        continue
    kaggle.api.competition_download_file(
        COMP, f'train_images/{pid}/{iid}.dcm', path=dst, quiet=True)
    _unzip_all(dst)

n_dcm = len(glob.glob(os.path.join(SAMPLE_DIR, 'train_images', '**', '*.dcm'), recursive=True))
print(f'{n_dcm} DICOM dans {SAMPLE_DIR} — prêts pour le chapitre 2.5.')

In [ ]:
# --- Jeu COMPLET (images DICOM, ~314 Go) : passer FULL=True pour télécharger ---
FULL = False
if FULL and not os.path.isdir(os.path.join(DATA_DIR, 'train_images')):
    kaggle.api.competition_download_files(COMP, path=DATA_DIR, quiet=False)
    _unzip_all(DATA_DIR)
    print('Téléchargement complet terminé.')
else:
    present = os.path.isdir(os.path.join(DATA_DIR, 'train_images'))
    print('train_images déjà présent :' if present else 'FULL=False (téléchargement complet ignoré).',
          present)